# Figure 2 — Distinct perturbation programs converge on failed endothelial differentiation

This notebook reproduces panels **a, b, d, e and f** and intentionally omits schematic panel c. Panels a, b, d and e use **plotnine**; panel f retains its existing Matplotlib renderer. Panels a–b read `data/figure_02/cNMF_50_0_2_032526_6co.xlsx`. Panel b retains the EC-output column and uses all **120 control guides** (60 non-targeting plus 60 safe-targeting), Welch tests, and BH correction across 297 targets.


In [ ]:
from IPython.display import display
import matplotlib.pyplot as plt
from src.cc_perturbseq.figure_notebook import get_path, initialize_figure, load_figure_manifest, panel_directory, sha256_file, write_receipt
from src.cc_perturbseq.figure2_panels import (
    FIGURE2D_TARGETS, FIGURE2_TARGET_GROUPS, compute_figure2b_ec_statistics,
    compute_d3_guide_ec_statistics, load_figure2_cnmf_workbook,
    plot_figure2a, plot_figure2b, plot_figure2d, plot_figure2e_umaps,
)
from src.cc_perturbseq.program_panels import carrier_seeded_neighborhood, load_guide_target_map, load_integrated_anndata, plot_carrier_neighborhood_grid

EXPECTED_SHA = "d2fe4834b1efd3c00e01062b56b907a6178bfbc48a40beb5ea7b347538bfc824"

def export_figure(obj, panel, stem):
    out = panel_directory(ctx, panel); result = {}
    is_plotnine = obj.__class__.__module__.startswith("plotnine")
    for suffix in ("pdf", "svg", "png"):
        path = out / f"{stem}.{suffix}"
        if is_plotnine:
            obj.save(filename=path, dpi=300, verbose=False)
        else:
            obj.savefig(path, dpi=300 if suffix == "png" else None, bbox_inches="tight", facecolor="white")
        result[suffix] = str(path)
    return result

FIGURE = load_figure_manifest("Figure_02")
FIGURE["panels"] = [p for p in FIGURE["panels"] if p["panel"] in {"a","b","d","e","f"}]
for p in FIGURE["panels"]:
    p["required_inputs"] = [] if p["panel"] == "a" else ["ccperturbseq.integrated_anndata","ccperturbseq.cnmf_usage","ccperturbseq.target_metadata"]
FIGURE["scope"] = "Panels a–b and d–f"
ctx = initialize_figure(FIGURE)

workbook = ctx["root"] / "data" / "figure_02" / "cNMF_50_0_2_032526_6co.xlsx"
if sha256_file(workbook) != EXPECTED_SHA:
    raise ValueError("Missing or incorrect repository-local Figure 2 cNMF workbook")
effect_matrix, selected_associations, workbook_provenance = load_figure2_cnmf_workbook(workbook)
adata_path = get_path(ctx["paths"], "ccperturbseq.integrated_anndata")
usage_path = get_path(ctx["paths"], "ccperturbseq.cnmf_usage")
guide_metadata_path = get_path(ctx["paths"], "ccperturbseq.target_metadata")
adata, usage = load_integrated_anndata(adata_path, usage_path)
ctx["input_audit"]


## Panel a — perturbation-effect correlations

Pearson correlations use 200 features per profile (50 programs × D0–D3), average linkage on \(1-r\), and optimal leaf ordering. `NEDD8-MDP1` is removed while canonical `NEDD8` and the pooled non-targeting profile are retained.


In [ ]:
plot_a, corr_a, order_a, inset_a, boxes_a = plot_figure2a(effect_matrix)
out = panel_directory(ctx, "a"); outputs = export_figure(plot_a, "a", "Figure_02a_perturbation_effect_correlation")
corr_a.to_csv(out / "Figure_02a_target_correlation.csv")
order_a.to_csv(out / "Figure_02a_clustered_target_order.csv", index=False)
inset_a.to_csv(out / "Figure_02a_selected_group_correlations.csv", index=False)
boxes_a.to_csv(out / "Figure_02a_cluster_boxes.csv", index=False)
write_receipt(ctx, "a", status="SOURCE_VERIFIED_IMPLEMENTATION", inputs={"workbook":str(workbook),"sha256":EXPECTED_SHA}, outputs=outputs, parameters={"profiles":"297 targets plus pooled control","features":200,"distance":"1-Pearson r","linkage":"average, optimal ordering","plotting_backend":"plotnine"})
display(plot_a)


## Panel b — program effects and Day 3 EC output

The EC-output column uses all 120 controls. Target-guide percentages exclude cells carrying any control guide; each control-guide percentage excludes cells carrying the evaluated target. Welch P values are BH-adjusted across all 297 targets.


In [ ]:
guide_b, stats_b, meta_b = compute_figure2b_ec_statistics(adata, guide_metadata_path)
plot_b, effects_b, fdr_b, program_source_b, ec_source_b = plot_figure2b(selected_associations, stats_b)
out = panel_directory(ctx, "b"); outputs = export_figure(plot_b, "b", "Figure_02b_selected_program_and_EC_effects")
effects_b.to_csv(out / "Figure_02b_effect_matrix.csv")
fdr_b.to_csv(out / "Figure_02b_FDR_matrix.csv")
program_source_b.to_csv(out / "Figure_02b_program_effect_source.csv", index=False)
stats_b.to_csv(out / "Figure_02b_Welch_EC_statistics_all_297_genes.csv", index=False)
guide_b.to_csv(out / "Figure_02b_guide_level_D3_EC_percentages_all_297_genes.csv.gz", index=False, compression="gzip")
write_receipt(ctx, "b", status="SOURCE_VERIFIED_METHOD_IMPLEMENTATION", inputs={"workbook":str(workbook),"anndata":str(adata_path),"guide_metadata":str(guide_metadata_path)}, outputs=outputs, parameters={"control_guides":"60 non-targeting + 60 safe-targeting","target_exclusion":"any control guide","control_exclusion":"evaluated target","test":"two-sided Welch t-test","BH_tests":297,"effect":"log2(target/control)","plotting_backend":"plotnine",**meta_b})
display(plot_b)
stats_b.set_index("gene").loc[[g for genes in FIGURE2_TARGET_GROUPS.values() for g in genes], ["mean_target_EC_pct","mean_control_EC_pct","welch_pvalue_two_sided","FDR_BH_297_genes","significance"]]


## Panel d — guide-level Day 3 EC differentiation

For each non-targeting guide, cells also carrying any guide targeting KDR, PLCG1, PRKCD or ETV2 are excluded. The four targets are compared with the resulting non-targeting-guide distribution using Welch tests and BH correction across four tests.


In [ ]:
guide_d, stats_d, meta_d = compute_d3_guide_ec_statistics(adata, guide_metadata_path, FIGURE2D_TARGETS, nt_exclusion_targets=FIGURE2D_TARGETS)
plot_d, jitter_d = plot_figure2d(guide_d, stats_d)
out = panel_directory(ctx, "d"); outputs = export_figure(plot_d, "d", "Figure_02d_guide_level_D3_EC_fraction")
guide_d.to_csv(out / "Figure_02d_guide_level_D3_EC_percentages.csv", index=False)
stats_d.to_csv(out / "Figure_02d_Welch_statistics.csv", index=False)
jitter_d.to_csv(out / "Figure_02d_plot_jitter_coordinates.csv", index=False)
write_receipt(ctx, "d", status="SOURCE_VERIFIED_METHOD_IMPLEMENTATION", inputs={"anndata":str(adata_path),"guide_metadata":str(guide_metadata_path)}, outputs=outputs, parameters={"targets":FIGURE2D_TARGETS,"control":"60 non-targeting guides","NT_exclusion_targets":FIGURE2D_TARGETS,"test":"two-sided Welch t-test","BH_tests":4,"plotting_backend":"plotnine",**meta_d})
display(plot_d)
stats_d


## Panel e — P17/P27 and marker UMAPs

P17, SLC8A1, P27 and GATA4 are shown with the prior `YlGnBu` scale, light-gray background, and feature-specific 99th-percentile maxima.


In [ ]:
plot_e, source_e = plot_figure2e_umaps(adata, usage)
out = panel_directory(ctx, "e"); outputs = export_figure(plot_e, "e", "Figure_02e_P17_P27_SLC8A1_GATA4_umaps")
source_paths = {}
for name, table in source_e.items():
    path = out / f"Figure_02e_{name}_source.csv.gz"; table.to_csv(path, compression="gzip"); source_paths[name] = str(path)
write_receipt(ctx, "e", status="SOURCE_STYLE_RECOVERED", inputs={"anndata":str(adata_path),"usage":str(usage_path)}, outputs={**outputs,**source_paths}, parameters={"features":["P17","SLC8A1","P27","GATA4"],"cmap":"YlGnBu","vmax":"feature-specific p99","plotting_backend":"plotnine"})
display(plot_e)


## Panel f — carrier-seeded neighborhood enrichment

The existing same-day 300-neighbor program-space implementation and Matplotlib renderer are retained.


In [ ]:
guide_map = load_guide_target_map(adata, guide_metadata_path)
targets_f = ["ETV2","KDR","PRKCD","PLCG1"]
neighborhoods_f = carrier_seeded_neighborhood(adata, usage, guide_map, targets_f, k=300, pseudocount=0.5)
fig_f = plot_carrier_neighborhood_grid(adata, neighborhoods_f, targets_f)
out = panel_directory(ctx, "f"); outputs = export_figure(fig_f, "f", "Figure_02f_carrier_seeded_neighborhoods")
neighborhoods_f.to_csv(out / "Figure_02f_neighborhood_statistics.csv", index=False)
write_receipt(ctx, "f", status="IMPLEMENTED_NOT_YET_NUMERICALLY_VALIDATED", inputs={"anndata":str(adata_path),"usage":str(usage_path),"guide_metadata":str(guide_metadata_path)}, outputs=outputs, parameters={"targets":targets_f,"k":300,"distance":"Euclidean on z-scored 50-program usage","same_day":true,"pseudocount":0.5,"plotting_backend":"Matplotlib"})
display(fig_f)
